In [6]:
import datetime as dt
import re
from datetime import date
import pandas as pd

def calc_rate(symbol_to, cost):
    if not str(symbol_to).strip():
        return
    symbolOp = re.split(r'([\d.]+)', symbol_to)
    final_value = float(symbolOp[3])
    final_date = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
    days_hold = (final_date - date.today()).days
    rate_of_gain = float(cost) / final_value * (365 / days_hold) * 100
    return str(round(rate_of_gain))
#print(calc_rate('GOOGL260618P377', 12.75))

df = pd.read_csv(r'E:\My Drive\tax\out_fidelity.csv', on_bad_lines='skip')
df = df[df['Ticker'].isin(['sell', 'buy'])]
#print (df[['Account', 'Symbol', 'sell']])
for i in df.index:
    try:
        df.loc[i, 'o_ratePercent'] = calc_rate(df['Symbol'][i], df['sell'][i])
    except Exception as e:
        print(i, e)
print(df[['Account', 'Ticker', 'Symbol', 'sell', 'o_ratePercent']])

       Account Ticker          Symbol   sell o_ratePercent
11   218320885   sell    AGQ260918C75   9.20           102
35   X65750304   sell  COIN260821C180  19.00           241
75   X83939133   sell  NVDA260821C215   7.90            84
92   218320886   sell   RKLB260918C85  13.40           131
96   X65750304   sell  SLV260821C56.5   2.24            90
97   X65750304   sell  SLV260918C56.5   3.70            54
125  218320886   sell    TLRY260918C5   0.45            75
128  218320886   sell   ARM260821P240  20.00           190
129  218320886   sell    MU260821P825  68.00           188


In [ ]:
def roll_gain(symbol_from, symbol_to, diff):
    symbolOp = re.split(r'([\d.]+)', symbol_from)
    initial_value = float(symbolOp[3])
    symbolOp = re.split(r'([\d.]+)', symbol_to)
    final_value = float(symbolOp[3])
    final_date = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
    days_hold = (final_date - date.today()).days
    rate_of_gain = (final_value - initial_value + float(diff)) / initial_value * (365 / days_hold) * 100
    return str(round(rate_of_gain))

df = pd.read_csv(r'G:\My Drive\data\out_fidelity.csv')
df = df[(df['gainLoss'] == 'roll')]

for i in df.index:
    try:
        df.loc[i, 'comments'] = roll_gain(df['Symbol'][i], df['TEMA_seq'][i], df['pct'][i])
    except Exception as e:
        print(i, e)
df[['Account', 'Symbol', 'sell', 'pct', 'TEMA_seq', 'comments']]

KeyError: 'gainLoss'

In [ ]:
print(calc_rate('COST800',31.75))
print(calc_rate('XOM115',4.95))
print(calc_rate('MSFT395',17.8))
print(calc_rate('ISRG440',21))
print(calc_rate('GOOGL160',8.35))
print(calc_rate('AAPL210',11.35))
print(calc_rate('MU103',5.5))
print(calc_rate('AMZN165',10.5))
print(calc_rate('ABNB125',9.35))
print(calc_rate('SQ57.5',4.6))
print(calc_rate('LVS40',3.35))
print(calc_rate('TSLA200',18.1))
print(calc_rate('CCL15',1.4))
print(calc_rate('CRWD230',25.45))
print(calc_rate('NVDA100',11.3))
print(calc_rate('SMCI620',77.5))
print(calc_rate('COIN195',25))
print(calc_rate('TQQQ54',7.55))
print(calc_rate('MSTR1320',200.6))
print(calc_rate('NNOX7.5',1.3))
print(calc_rate('SOXL28',6.3))
print(calc_rate('DJT27.5',6.8))

In [ ]:
roll_gain('UAL240315C39', 4.9, 5.1, 'UAL240419C40')

In [ ]:
def calculate_rate_of_gain(initial_value, final_value, symbol):
    symbolOp = re.split(r'([\d.]+)', symbol)
    myDate = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
    days_hold = (myDate - date.today()).days
    rate_of_gain = (final_value - initial_value) / initial_value * (365 / days_hold) * 100
    return round(rate_of_gain)

calculate_rate_of_gain(100, 110, 'MSTR240419C940')

In [ ]:
def calculate_rate_of_gain(initial_value, final_value, days_hold):
    rate_of_gain = (final_value - initial_value) / initial_value * (365 / days_hold) * 100
    return round(rate_of_gain, 1)

calculate_rate_of_gain(100, 110, 30)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf


def plot_covered_call(symbol: str, strike: float, premium: float, cost_basis: float = None):
    if cost_basis is None:
        last = getattr(yf.Ticker(symbol).fast_info, 'last_price', None)
        if not last:
            raise RuntimeError(f"Could not fetch current price for {symbol}")
        cost_basis = round(last, 2)

    breakeven  = cost_basis - premium
    max_profit = strike - cost_basis + premium

    print(f"Cost basis: ${cost_basis:.2f}")
    print(f"Breakeven:  ${breakeven:.2f}")
    print(f"Max profit: ${max_profit:.2f}  (at or above ${strike})")

    S = np.linspace(strike - 14, strike + 14, 500)

    long_stock = S - cost_basis
    short_call = np.where(S <= strike, premium, premium - (S - strike))
    combined   = long_stock + short_call

    _, ax = plt.subplots(figsize=(10, 6))

    ax.plot(S, long_stock, '--', color='steelblue',  alpha=0.7, label=f'Long Stock (cost ${cost_basis:.2f})')
    ax.plot(S, short_call, '--', color='darkorange',  alpha=0.7, label=f'Short Call (strike ${strike}, premium ${premium})')
    ax.plot(S, combined,         color='green',       linewidth=2.5, label='Covered Call (combined)')

    ax.axhline(0,         color='black', linewidth=0.8)
    ax.axvline(strike,    color='gray',  linestyle=':',  linewidth=1.2, label=f'Strike ${strike}')
    ax.axvline(breakeven, color='red',   linestyle='--', linewidth=1.2, label=f'Breakeven ${breakeven:.2f}')

    ax.fill_between(S, combined, 0, where=(combined >= 0), color='green', alpha=0.08)
    ax.fill_between(S, combined, 0, where=(combined <  0), color='red',   alpha=0.08)

    ax.set_xlabel(f'{symbol} Price at Expiration ($)', fontsize=12)
    ax.set_ylabel('Profit / Loss ($)',                 fontsize=12)
    ax.set_title(f'Covered Call P&L — {symbol} | Strike ${strike} | Premium ${premium}', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_long_itm_call(symbol: str, strike: float, premium: float, cost_basis: float = None):
    """Buy deep ITM call as stock replacement — shows downside protection vs owning stock."""
    if cost_basis is None:
        last = getattr(yf.Ticker(symbol).fast_info, 'last_price', None)
        if not last:
            raise RuntimeError(f"Could not fetch current price for {symbol}")
        cost_basis = round(last, 2)

    breakeven = strike + premium
    cost = breakeven - cost_basis
    print(f"Cost basis (stock): ${cost_basis:.2f}")
    print(f"Strike:             ${strike:.2f}")
    print(f"Premium paid:       ${premium:.2f}")
    print(f"Breakeven:          ${breakeven:.2f}")
    print(f"cost:               ${cost:.2f}")
    print(f"Max loss (call):    ${-premium:.2f}  (stock falls below ${strike})")

    S = np.linspace(strike - premium - 10, strike + premium + 20, 500)

    long_stock    = S - cost_basis
    long_itm_call = np.where(S <= strike, -premium, S - strike - premium)
    difference    = long_itm_call - long_stock

    _, ax = plt.subplots(figsize=(10, 6))

    ax.plot(S, long_stock,    '--', color='steelblue', alpha=0.7, linewidth=1.8,
            label=f'Long Stock (cost ${cost_basis:.2f})')
    ax.plot(S, long_itm_call,       color='green',     linewidth=2.5,
            label=f'Long ITM Call (strike ${strike}, premium ${premium})')


    ax.axhline(0,         color='black', linewidth=0.8)
    ax.axvline(strike,    color='gray',  linestyle=':',  linewidth=1.2, label=f'Strike ${strike}')
    ax.axvline(breakeven, color='red',   linestyle='--', linewidth=1.2, label=f'Breakeven ${breakeven:.2f}')
    ax.axhline(-premium,  color='orange', linestyle=':', linewidth=1.2, label=f'Cost −${cost:.2f} Max loss −${premium}')

    ax.fill_between(S, long_itm_call, 0, where=(long_itm_call >= 0), color='green', alpha=0.08)
    ax.fill_between(S, long_itm_call, 0, where=(long_itm_call <  0), color='red',   alpha=0.08)
    
    ax.set_xlabel(f'{symbol} Price at Expiration ($)', fontsize=12)
    ax.set_ylabel('Profit / Loss ($)',                 fontsize=12)
    ax.set_title(
        f'Deep ITM Call as Stock Replacement — {symbol} | Strike ${strike} | Premium ${premium}',
        fontsize=13
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == '__main__':
    #plot_covered_call(symbol='HPE', strike=38.0, premium=2.7)
    plot_long_itm_call(symbol='AMD', strike=435, premium=52)
